# ConvGeM-NeXt — run every experiment on Colab

Runtime → Change runtime type → **GPU**. Run one step per session if needed; `runs/` and `data/` live on Google Drive, so finished steps survive disconnects. Full guide: `EXPERIMENTS.md`.

## 1. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/ConvGeM-NeXt-runs'   # data + results are kept here
import os; os.makedirs(WORK, exist_ok=True)

!git clone -q https://github.com/HAWKAARJAV/ConvGeM-NeXt.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/ConvGeM-NeXt
!pip -q install -r requirements.txt kaggle

# keep data/ and results/ on Drive
for d in ['data', 'runs']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        !cp -rn {d}/. {WORK}/{d}/ && rm -rf {d}
    if not os.path.exists(d):
        os.symlink(f'{WORK}/{d}', d)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Kaggle token (for PlantVillage)
Upload `kaggle.json` (kaggle.com → Settings → API → Create New Token).

In [ ]:
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    files.upload()   # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 3. Datasets → `data/` (download, arrange, verify, create splits)

In [ ]:
!bash run_experiments.sh data
!head -20 data/dataset_report.md

## 4. Experiments
Each cell is one step. Settings can be overridden, e.g. `!EPOCHS=10 bash run_experiments.sh e1`.

In [ ]:
# E2 — PlantDoc ablation, 4 variants (~1–1.5 h on T4)
!bash run_experiments.sh e2

In [ ]:
# E1 — PlantVillage ablation, 4 variants (long: ~15–20 h on T4 at 30 epochs; consider EPOCHS=10 or an A100)
!bash run_experiments.sh e1

In [ ]:
# E3 + E4 — cross-corpus zero-shot (needs E1 and E2; ~15 min)
!bash run_experiments.sh e3
!bash run_experiments.sh e4

In [ ]:
# E5 — training improvements S0–S6 on the proposed model (needs E1; ~1.5–2 h)
!bash run_experiments.sh e5

## 5. Figures and the table of all measured numbers

In [ ]:
!bash run_experiments.sh figures
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('runs/figures/*.png')):
    print(f); display(Image(f, width=700))
import pandas as pd
pd.read_csv('runs/figures/all_results.csv')

## 6. Download results
Everything is already on Drive (`MyDrive/ConvGeM-NeXt-runs/runs`). Optional zip without checkpoints:

In [ ]:
!cd runs && zip -qr /content/results_no_ckpt.zip . -x '*.pth'
from google.colab import files; files.download('/content/results_no_ckpt.zip')